# NILM-Engine — Ablation 실험 노트북

학습(`colab_gcs_train.ipynb`)과 분리된 단독 ablation 실행 파일.  
각 셀은 독립적으로 실행 가능 — 캐시 빌드와 비교 분석을 별도로 제어한다.

| 실험 | 변수 | 위치 |
|------|------|------|
| A-0 | Denoise ON vs OFF | 이 노트북 |

**실행 순서**: 1(환경) → 2(GCS) → 3(상수) → 4(리포지토리) → 5(Drive) → 6(캐시) → 7(학습) → 8(비교)

## 1. 환경 설치 & GCS 인증

In [ ]:
!pip install -q gudhi
from google.colab import auth
auth.authenticate_user()
print('GCS 인증 완료')

## 2. GCS 연결

In [ ]:
import gcsfs
gcs = gcsfs.GCSFileSystem()

# 연결 검증
!gcloud storage ls gs://ax-nilm-data-dhwang0803-us/nilm/training_dev10/ | head -3
print('GCS 연결 완료')

## 3. 상수 정의

In [ ]:
SPLIT = {
    'train': ['house_011', 'house_015', 'house_016', 'house_017',
              'house_033', 'house_039', 'house_054', 'house_063'],
    'val':   ['house_049'],
    'test':  ['house_067'],
}
BUCKET_PREFIX = 'ax-nilm-data-dhwang0803-us/nilm/training_dev10'
LABEL_PATH    = 'ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet'

# ablation 대상 — 변경 시 이 셀만 수정
ABL_EXP   = 'EXP1'
ABL_MODEL = 'cnn_tda'

print(f'ablation: {ABL_EXP} / {ABL_MODEL}')

## 4. 리포지토리 설정

In [ ]:
import os, sys, yaml

REPO_DIR = '/content/ax_nilm'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print('클론 완료')
else:
    !git -C {REPO_DIR} pull -q && echo '최신화 완료'

SRC_DIR     = f'{REPO_DIR}/nilm-engine/src'
SCRIPTS_DIR = f'{REPO_DIR}/nilm-engine/scripts'
CFG_DIR     = f'{REPO_DIR}/nilm-engine/config'

for d in [SRC_DIR, SCRIPTS_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)

from acquisition.gcs_loader import GCSNILMDataset
from acquisition.preprocessor import PowerScaler
from train_model import (
    build_model, evaluate, train_one_epoch,
    compute_pos_weight, _NILMDatasetWithTDA, APPLIANCE_LABELS,
)

with open(f'{CFG_DIR}/train.yaml')   as f: TRAIN_CFG   = yaml.safe_load(f)
with open(f'{CFG_DIR}/dataset.yaml') as f: DATASET_CFG = yaml.safe_load(f)

print('모듈 import 완료')

## 5. Google Drive 마운트

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

CKPT_DIR    = Path('/content/drive/MyDrive/nilm/checkpoints')
RESULTS_DIR = Path('/content/drive/MyDrive/nilm/results')
CACHE_DIR   = Path('/content/drive/MyDrive/nilm/cache')
for d in [CKPT_DIR, RESULTS_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'체크포인트 → {CKPT_DIR}')
print(f'결과       → {RESULTS_DIR}')
print(f'캐시       → {CACHE_DIR}')

## 6. 캐시 빌드 — CPU 런타임에서 실행

> **런타임 전환 가이드**
> - **셀 1~6 (여기까지)**: Colab 런타임 유형 → **CPU** (GPU 할당량 절약)
> - **셀 7~8 (학습·비교)**: 런타임 유형 → **GPU** 로 전환 후 셀 1~5 재실행 → 셀 7 실행
>   - Drive는 자동 마운트 유지되므로 캐시 재빌드 불필요

`denoise=True` 캐시가 `colab_gcs_train.ipynb`에서 이미 빌드됐다면 해당 줄을 주석 처리하면 된다.

In [ ]:
import gc

_ws   = DATASET_CFG['window']['size']
_st   = DATASET_CFG['window']['stride']
_ec   = DATASET_CFG['window'].get('event_context')
_ss   = DATASET_CFG['window'].get('steady_stride')
_week = TRAIN_CFG['experiments'][ABL_EXP].get('week')

def build_cache(denoise: bool):
    label = 'ON' if denoise else 'OFF'
    print(f'\n--- denoise={label} 캐시 빌드 ---')
    for split, houses in [('train', SPLIT['train']), ('val', SPLIT['val'])]:
        base = GCSNILMDataset(
            houses, gcs_fs=gcs,
            bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
            window_size=_ws, stride=_st, week=_week,
            event_context=_ec, steady_stride=_ss,
            fit_scaler=(split == 'train'),
            cache_dir=CACHE_DIR,
            denoise=denoise,
        )
        print(f'  {split}: {len(base):,} windows')
        if ABL_MODEL == 'cnn_tda':
            tda = _NILMDatasetWithTDA(base, cache_dir=CACHE_DIR)
            print(f'  {split} TDA: 완료')
            del tda
        del base
        gc.collect()

# 필요한 조건만 주석 해제해서 실행
build_cache(denoise=True)   # train.ipynb 셀-7에서 이미 했으면 주석 처리
build_cache(denoise=False)

print('\n캐시 완료')

## 7. 학습 — denoise ON / OFF

두 조건을 순차 실행. 체크포인트/메트릭은 tag suffix로 분리 저장된다:
- `EXP1_cnn_tda_denoise_on.pt`
- `EXP1_cnn_tda_denoise_off.pt`

In [ ]:
import json, time
import torch
from torch.utils.data import DataLoader


def _run_ablation_condition(denoise: bool, tag: str) -> dict:
    """단일 denoise 조건 학습. run_exp_gcs 로직과 동일, tag로 파일 분리."""
    exp_cfg    = TRAIN_CFG['experiments'][ABL_EXP]
    week       = exp_cfg.get('week')
    ws         = DATASET_CFG['window']['size']
    st         = DATASET_CFG['window']['stride']
    ec         = DATASET_CFG['window'].get('event_context')
    ss         = DATASET_CFG['window'].get('steady_stride')
    bs         = TRAIN_CFG['training']['batch_size']
    ep         = TRAIN_CFG['training']['epochs']
    pat        = TRAIN_CFG['training']['early_stopping_patience']
    lr         = TRAIN_CFG['training']['learning_rate']
    wd         = TRAIN_CFG['optimizer']['weight_decay']
    lambda_mse = TRAIN_CFG['training']['loss_weights']['mse']

    suffix = f'_{tag}'
    print(f"\n{'='*58}")
    print(f'  {ABL_EXP}/{ABL_MODEL}[{tag}]  week={week}  denoise={denoise}')
    print(f"{'='*58}")

    ds_kwargs = dict(
        gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX,
        label_path=LABEL_PATH,
        window_size=ws, stride=st, week=week,
        event_context=ec, steady_stride=ss,
        cache_dir=CACHE_DIR,
        denoise=denoise,
    )
    base_train = GCSNILMDataset(SPLIT['train'], fit_scaler=True,            **ds_kwargs)
    base_val   = GCSNILMDataset(SPLIT['val'],   scaler=base_train.scaler,   **ds_kwargs)

    if ABL_MODEL == 'cnn_tda':
        train_ds = _NILMDatasetWithTDA(base_train, cache_dir=CACHE_DIR)
        val_ds   = _NILMDatasetWithTDA(base_val,   cache_dir=CACHE_DIR)
    else:
        train_ds, val_ds = base_train, base_val

    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=False)
    val_dl   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)
    print(f'  train={len(train_ds):,}  val={len(val_ds):,} windows')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = build_model(ABL_MODEL, ws).to(device)

    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min',
        factor=TRAIN_CFG['scheduler']['factor'],
        patience=TRAIN_CFG['scheduler']['patience'],
    )
    amp_scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    pos_weight = None
    if ABL_MODEL == 'cnn_tda':
        print('  pos_weight 계산 중...')
        pos_weight = compute_pos_weight(train_dl, device)

    best_mae, best_vm, best_state, no_improve = float('inf'), None, None, 0
    t0 = time.perf_counter()

    for epoch in range(1, ep + 1):
        loss = train_one_epoch(model, train_dl, optimizer, ABL_MODEL, device,
                               amp_scaler=amp_scaler, pos_weight=pos_weight,
                               lambda_mse=lambda_mse)
        vm = evaluate(model, val_dl, ABL_MODEL, device)
        scheduler.step(vm['mae'])
        lr_now = optimizer.param_groups[0]['lr']
        f1c = f"  f1_cls={vm['f1_cls']:.3f}" if vm.get('f1_cls') is not None else ''
        print(f"  ep {epoch:3d}/{ep}  loss={loss:.4f}  "
              f"val_mae={vm['mae']:.2f}W  f1={vm['f1']:.3f}{f1c}  lr={lr_now:.2e}")

        if vm['mae'] < best_mae - 1e-4:
            best_mae, best_vm = vm['mae'], vm
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= pat:
                print(f'  조기 종료 ({pat} epoch 개선 없음)')
                break

    elapsed = time.perf_counter() - t0

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    torch.save(model.state_dict(), CKPT_DIR / f'{ABL_EXP}_{ABL_MODEL}{suffix}.pt')
    if base_train.scaler is not None:
        base_train.scaler.save(CKPT_DIR / f'{ABL_EXP}_{ABL_MODEL}{suffix}_scaler.json')

    fm = best_vm if best_vm is not None else vm
    fm.update({'exp': ABL_EXP, 'model': ABL_MODEL, 'tag': tag,
               'denoise': denoise, 'training_time_s': round(elapsed, 1)})
    with open(RESULTS_DIR / f'{ABL_EXP}_{ABL_MODEL}{suffix}_metrics.json', 'w') as f:
        json.dump(fm, f, ensure_ascii=False, indent=2)

    f1c = f"  F1_cls={fm['f1_cls']:.3f}" if fm.get('f1_cls') is not None else ''
    print(f"  ✅ MAE={fm['mae']:.2f}W  F1={fm['f1']:.3f}{f1c}  ({elapsed:.0f}s)")
    return fm


m_on  = _run_ablation_condition(denoise=True,  tag='denoise_on')
m_off = _run_ablation_condition(denoise=False, tag='denoise_off')

## 8. 결과 비교

학습을 다시 돌리지 않고 저장된 메트릭만 로드해 비교할 때도 이 셀만 실행하면 된다.

In [ ]:
import json, shutil
from pathlib import Path

# 저장된 파일에서 로드 (학습 셀을 건너뛰고 이 셀만 실행해도 동작)
_p_on  = RESULTS_DIR / f'{ABL_EXP}_{ABL_MODEL}_denoise_on_metrics.json'
_p_off = RESULTS_DIR / f'{ABL_EXP}_{ABL_MODEL}_denoise_off_metrics.json'

if not _p_on.exists():
    raise FileNotFoundError(f'메트릭 파일 없음: {_p_on}  → 셀-7을 먼저 실행하세요.')

m_on  = json.load(open(_p_on))
m_off = json.load(open(_p_off))

# ── 전체 지표 비교 ────────────────────────────────────────────────────────────
def fmt(v):
    return f'{v:.3f}' if v is not None else '  —  '

print('=' * 62)
print(f'  Denoise Ablation — {ABL_EXP} / {ABL_MODEL}')
print('=' * 62)
print(f"\n{'지표':<12} {'denoise=ON':>12} {'denoise=OFF':>12} {'차이(ON-OFF)':>14}")
print('-' * 54)
for key, label in [('f1', 'F1(전체)'), ('f1_cls', 'F1_cls'),
                   ('mae', 'MAE(W)'), ('rmse', 'RMSE(W)'), ('sae', 'SAE')]:
    v_on  = m_on.get(key)
    v_off = m_off.get(key)
    diff  = f'{v_on - v_off:+.3f}' if (v_on is not None and v_off is not None) else '—'
    print(f'  {label:<10} {fmt(v_on):>12} {fmt(v_off):>12} {diff:>14}')

# ── per-appliance F1 비교 ─────────────────────────────────────────────────────
GROUPS = {
    'fast transient (손상 의심 대상)': [
        '전기포트', '헤어드라이기', '선풍기', '에어프라이어',
        '인덕션(전기레인지)', '전기다리미', '진공청소기(유선)',
    ],
    'cycle (노이즈 민감, denoising 수혜 가능)': [
        '세탁기', '에어컨', '의류건조기', '식기세척기/건조기', '전기밥솥',
    ],
    'always-on / low-power': [
        '공기청정기', '제습기', '일반 냉장고', '김치냉장고', '무선공유기/셋톱박스',
    ],
}

for group_label, names in GROUPS.items():
    print(f'\n  ── {group_label}')
    print(f"  {'가전':<22} {'F1 ON':>8} {'F1 OFF':>8} {'차이':>8} {'판정':>10}")
    print('  ' + '-' * 58)
    for name in names:
        f_on  = m_on.get('per_appliance',  {}).get(name, {}).get('f1')
        f_off = m_off.get('per_appliance', {}).get(name, {}).get('f1')
        if f_on is None and f_off is None:
            print(f'  {name:<22} {"—":>8} {"—":>8} {"—":>8} {"데이터없음":>10}')
            continue
        diff = (f_on - f_off) if (f_on is not None and f_off is not None) else None
        if diff is None:      judge = '—'
        elif diff >  0.02:    judge = 'ON 우세'
        elif diff < -0.02:    judge = '⚠️ OFF 우세'
        else:                 judge = '동등'
        diff_str = f'{diff:+.3f}' if diff is not None else '—'
        print(f'  {name:<22} {fmt(f_on):>8} {fmt(f_off):>8} {diff_str:>8} {judge:>10}')

# ── 종합 판정 ─────────────────────────────────────────────────────────────────
f1_on  = m_on.get('f1', 0)
f1_off = m_off.get('f1', 0)
print('\n판정 기준:')
print('  ON 우세  (diff > +0.02) → denoising 유지')
print('  동등     (|diff| ≤ 0.02) → 효과 없음. OFF 고려 (학습 속도 개선)')
print('  ⚠️ OFF 우세 (diff < -0.02) → transient 손상 의심. denoising 제거 권장')
print(f'\n  → 전체 F1 차이: {f1_on - f1_off:+.3f}')

# 비교 결과 저장
abl_result = {'denoise_on': m_on, 'denoise_off': m_off}
out = RESULTS_DIR / f'{ABL_EXP}_{ABL_MODEL}_denoise_ablation.json'
with open(out, 'w') as f:
    json.dump(abl_result, f, ensure_ascii=False, indent=2)
print(f'\n비교 결과 저장: {out.name}')

# ── 승자를 EXP1_cnn_tda.pt 로 승격 → train 노트북은 EXP2부터 시작 ─────────────
winner = 'denoise_on' if f1_on >= f1_off else 'denoise_off'
print(f'\n{"="*62}')
print(f'  승격: {winner}  →  {ABL_EXP}_{ABL_MODEL}.pt')
print(f'{"="*62}')

for src_path, dst_path in [
    (CKPT_DIR    / f'{ABL_EXP}_{ABL_MODEL}_{winner}.pt',
     CKPT_DIR    / f'{ABL_EXP}_{ABL_MODEL}.pt'),
    (CKPT_DIR    / f'{ABL_EXP}_{ABL_MODEL}_{winner}_scaler.json',
     CKPT_DIR    / f'{ABL_EXP}_{ABL_MODEL}_scaler.json'),
    (RESULTS_DIR / f'{ABL_EXP}_{ABL_MODEL}_{winner}_metrics.json',
     RESULTS_DIR / f'{ABL_EXP}_{ABL_MODEL}_metrics.json'),
]:
    if src_path.exists():
        shutil.copy(src_path, dst_path)
        print(f'  {src_path.name}  →  {dst_path.name}')
    else:
        print(f'  ⚠️ {src_path.name} 없음 — 건너뜀')

print(f'\n✅ train 노트북은 EXP2부터 실행하세요 (resume_from: EXP1 설정 확인)')
